# Scoring Tutorial


## Step 1: Import Libraries


In [1]:
from notebook_imports import *

/Users/matthewho/miniconda3/envs/arc_agi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from itertools import combinations

from concept_mem.evaluation.score_tree import (
    flatten_solution_trees,
    official_score,
    official_score_per_puzzle,
    strict_score,
    strict_score_per_step,
    strict_score_per_puzzle,
)
from concept_mem.evaluation.solution_tree import (
    create_solution_tree_from_serialized_dict,
)

## Step 2: Load Data and Convert Result Directories to Dataframes


In [3]:
# load target problems
val100 = load_arc_data("val100")

In [4]:
# specify output directories
OUTPUTS_DIR = REPO_ROOT / "outputs"
res_directories = {
    "baseline0": OUTPUTS_DIR / "2025-06-29/16-45-06",
    "baseline1": OUTPUTS_DIR / "2025-07-02/16-17-40",
    "baseline2": OUTPUTS_DIR / "2025-07-02/15-04-34",
}

In [5]:
# convert results to dataframes
res_dataframes = {k: result_dir_to_df(v)[0] for k, v in res_directories.items()}

In [6]:
# each dataframe has a row per test case result
# almost all columns are identifying what puzzle, what code gen attempt, and what test case
# then there's the "correct" column which is True if the test case was solved correctly
next(iter(res_dataframes.values()))

,puzzle_id,branch_id,thread_id,step_idx,is_train,case_idx,correct
0,97239e3d,0,0,0,True,0,True
1,97239e3d,0,0,0,True,1,False
2,97239e3d,0,0,0,True,2,False
3,97239e3d,0,0,0,False,0,False
4,97239e3d,0,0,1,True,0,True
...,...,...,...,...,...,...,...
836,9b4c17c4,0,0,0,False,0,True
837,9b4c17c4,0,0,0,False,1,True
838,070dd51e,0,0,0,True,0,True
839,070dd51e,0,0,0,True,1,True


## Step 3: Aggregate Token Usage

(across different runs for the same setting)


In [7]:
def _token_usage_from_run_dir(res_dir: Path) -> dict:
    # yields {iter_num: tok_usg_dict} per iteration
    aggregated_token_usage = {}
    for iteration_dir in res_dir.glob("iteration_*"):
        tok_usg_path = iteration_dir / "token_usage.json"
        if not tok_usg_path.exists():
            print(f"Token usage file not found in {iteration_dir}. Skipping.")
            continue
        m = re.search(r"iteration_(\d)", iteration_dir.name)
        if m is None:
            print(
                f"Could not extract iteration number from {tok_usg_path.name}. Skipping."
            )
            continue
        iteration_num = int(m.group(1))
        tok_usg = read_json(tok_usg_path)
        aggregated_token_usage[iteration_num] = tok_usg["after"]
    return aggregated_token_usage


def _combine_token_usage_dicts(
    agg: dict[str, dict[str, int]], new_usg: dict[str, dict[str, int]]
) -> None:
    for model, usage in new_usg.items():
        if model not in agg:
            agg[model] = defaultdict(int)
        for cat, count in usage.items():
            agg[model][cat] += count


def aggregate_token_usage(per_dir_usage: list[dict]) -> dict:
    max_iter_num = max([max(d.keys()) for d in per_dir_usage])
    aggregated_token_usage = defaultdict(dict)
    for i in range(1, max_iter_num + 1):
        # maps model -> usage
        # where usage is str -> int
        aggregated_token_usage[i] = defaultdict(lambda: defaultdict(int))
        for tok_usg in per_dir_usage:
            if i not in tok_usg:
                continue
            _combine_token_usage_dicts(aggregated_token_usage[i], tok_usg[i])
    return aggregated_token_usage


def aggregate_token_usage_from_dirs(run_dirs: list[Path]) -> dict:
    per_dir_usage = [_token_usage_from_run_dir(rd) for rd in run_dirs]
    return aggregate_token_usage(per_dir_usage)


def _tok_usg_from_cont_run(run_dir: Path) -> dict:
    aggregated_token_usage = {}
    for iter_dir in run_dir.glob("iter_*"):
        max_batch = -1
        max_batch_dir = None
        for batch_dir in iter_dir.glob("batch_*"):
            m = re.search(r"batch_(\d+)", batch_dir.name)
            batch_num = int(m.group(1))
            if batch_num > max_batch:
                max_batch = batch_num
                max_batch_dir = batch_dir
        iter_num = int(re.search(r"iter_(\d+)", iter_dir.name).group(1))
        tok_usg = read_json(max_batch_dir / "token_usage.json")
        aggregated_token_usage[iter_num] = tok_usg["after"]
    return aggregated_token_usage

In [8]:
baseline_token_usage = aggregate_token_usage_from_dirs(list(res_directories.values()))

## Step 4: Score Results


In [11]:
# for each of the 3 way solves build a df:
# - rows: iteration 1, 2, 3
# - columns: run0, run1, run2, singe_run_avg, single_run_std, run0and1, run1and2, run0and2, two_run_avg, two_run_std
def create_multi_run_summary(
    prefix: str,
    runs: list[Path] | list[pd.DataFrame],
    uid_subset: list[str] | None = None,
    official_or_strict: str = "official",
    **kwargs,
) -> pd.DataFrame:
    fixed_runs = []
    # uid_subset = set(uid_subset) if uid_subset is not None else None
    for run in runs:
        if isinstance(run, Path):
            run_df, _ = result_dir_to_df(run)
            run = run_df
        if uid_subset is not None:
            run = run[run["puzzle_id"].isin(uid_subset)].copy()
        fixed_runs.append(run)
    runs = fixed_runs

    # get official score per iteration for each individual run
    cols = {}
    for i, run in enumerate(runs):
        cols[f"run{i}"] = get_agg_score_per_iteration(
            run, prefix, official_or_strict, **kwargs
        )

    # get official score per best of 2 ensemble
    for i, j in combinations(range(len(runs)), 2):
        run0 = runs[i]
        run1 = runs[j]
        run0_duplicate = run0.copy()
        run0_duplicate["thread_id"] = 0
        run1_duplicate = run1.copy()
        run1_duplicate["thread_id"] = 1
        duplicate_df = pd.concat([run0_duplicate, run1_duplicate], ignore_index=True)
        ensemble_scores = get_agg_score_per_iteration(
            duplicate_df, prefix, official_or_strict, **kwargs
        )
        cols[f"ensemble_{i}_{j}"] = ensemble_scores

    # get official score of all 3 runs combined
    for i in range(3):
        runs[i]["thread_id"] = str(i)
    combined_df = pd.concat(runs, ignore_index=True)
    combined_scores = get_agg_score_per_iteration(
        combined_df, prefix, official_or_strict, **kwargs
    )
    cols["ensemble_all"] = combined_scores

    # combine into a DataFrame
    summary_df = pd.DataFrame(cols)
    # calculate averages and stds for each run
    sr_cols = [c for c in summary_df.columns if "run" in c]
    en_cols = [c for c in summary_df.columns if "ensemble" in c and "all" not in c]
    sr_df = summary_df[sr_cols]
    en_df = summary_df[en_cols]
    summary_df["single_run_avg"] = sr_df.mean(axis=1)
    summary_df["single_run_std"] = sr_df.std(axis=1)
    summary_df["two_run_avg"] = en_df.mean(axis=1)
    summary_df["two_run_std"] = en_df.std(axis=1)

    return summary_df.T


def get_agg_score_per_iteration(
    case_df: pd.DataFrame, prefix: str, official_or_strict: str = "official", **kwargs
) -> dict[str, float]:
    # get max iteration
    max_iteration = case_df["step_idx"].max()
    res = {}
    for i in range(max_iteration + 1):
        filtered_df = case_df[case_df["step_idx"] <= i]
        if official_or_strict == "official":
            iter_scores = official_score(filtered_df, step_selection="last", **kwargs)
        else:
            iter_scores = strict_score(filtered_df, step_selection="last", **kwargs)
        res[f"{prefix}_i{i}"] = iter_scores
    return res


# reformat sumdft, currently index contains "{settin_name}_i{iteration}", want multiindex with first level being setting name and second level being iteration
setting_iteration_pattern = re.compile(r"^(.*?)_i(\d)$")


def get_setting_name_and_iteration(combined_string: str) -> tuple[str, int]:
    match = setting_iteration_pattern.match(combined_string)
    if match:
        setting_name, iteration = match.groups()
        return setting_name, int(iteration)
    else:
        raise ValueError(
            f"String '{combined_string}' does not match the expected pattern."
        )


def create_combined_summary_df(
    run_groups: dict[str, list[pd.DataFrame]],
    id_subset: list[str] | None = None,
    official_or_strict: str = "official",
    **kwargs,
) -> pd.DataFrame:
    group_summaries = []
    for prefix, runs in run_groups.items():
        group_summary = create_multi_run_summary(
            prefix,
            runs,
            uid_subset=id_subset,
            official_or_strict=official_or_strict,
            **kwargs,
        )
        group_summaries.append(group_summary)
    combined_summary = pd.concat(group_summaries, axis=1)

    # reorder columns
    # target order: [run0, ..., ensemble_0, ..., single_run_avg, two_run_avg, ensemble_all, single_run_std, two_run_std]
    sumdft = combined_summary.T
    col_order = (
        [f"run{i}" for i in range(3)]
        + [f"ensemble_{i}_{j}" for i, j in combinations(range(3), 2)]
        + [
            "single_run_avg",
            "two_run_avg",
            "ensemble_all",
            "single_run_std",
            "two_run_std",
        ]
    )
    sumdft = sumdft[col_order]

    sumdft.index = pd.MultiIndex.from_tuples(
        [get_setting_name_and_iteration(idx) for idx in sumdft.index],
        names=["setting", "iteration"],
    )

    return sumdft

In [12]:
# create run groups
run_groups = {
    "baseline": list(res_dataframes.values()),
}

# expected input format:
# dict[str, list[pd.DataFrame]]
# mapping a setting name to a list of dataframes corresponding to different runs of that setting

official_score_summary = create_combined_summary_df(
    run_groups,
    id_subset=None,
    official_or_strict="official",
)

In [13]:
official_score_summary

run0  run1  run2  ensemble_0_1  ensemble_0_2  \
setting  iteration                                                 
baseline 0          46.0  45.5  47.5          51.5          57.0   
         1          57.0  55.5  61.0          62.5          70.0   
         2          59.5  59.0  65.0          66.0          70.0   

                    ensemble_1_2  single_run_avg  two_run_avg  ensemble_all  \
setting  iteration                                                            
baseline 0                  57.0       46.333333    55.166667          59.5   
         1                  67.5       57.833333    66.666667          71.5   
         2                  71.0       61.166667    69.000000          73.0   

                    single_run_std  two_run_std  
setting  iteration                               
baseline 0                1.040833     3.175426  
         1                2.843120     3.818813  
         2                3.329164     2.645751